In [1]:
import pandas as pd

with open('common_words.txt', 'r') as f:
    lines = f.readlines()
common_words = [line.strip() for line in lines if not line.startswith('#')][:1000]


py_df = pd.DataFrame({'rank': range(len(common_words)), 'username/repo_name': common_words})

all_safetensor_keys = set(f'{row["rank"]}.{row["username/repo_name"]}' for _, row in py_df.iterrows())
all_safetensor_keys

FileNotFoundError: [Errno 2] No such file or directory: 'common_words.txt'

In [ ]:
from safetensors.torch import load_file
import polars as pl
from pathlib import Path
import torch


schema = [('expert', pl.Int8)]
for _, row in py_df.iterrows():
    repo = row["username/repo_name"]
    # schema.append((f'{repo}.w1', pl.Float16))
    schema.append((f'{repo}.w3', pl.Float16))


layer_to_df: dict[int, pl.DataFrame] = {}

for layer in range(32):

    df = pl.DataFrame(schema=schema)
    layer_path = Path(f'output/common_words_1k/layer={layer:02d}')

    for expert in range(8):
        repos_to_activations = {}
        expert_path = layer_path / f'expert={expert}'

        w1_path = expert_path / 'w=1.safetensors'
        w3_path = expert_path / 'w=3.safetensors'

        w1 = load_file(w1_path)
        w3 = load_file(w3_path)

        assert set(w1.keys()) == all_safetensor_keys, f"Repo missing in w=1 for expert {expert}"
        assert set(w3.keys()) == all_safetensor_keys, f"Repo missing in w=3 for expert {expert}"

        for _, row in py_df.iterrows():
            key = f'{row["rank"]}.{row["username/repo_name"]}'

            w1_activation = w1[key].cpu().to(torch.float16).numpy()
            assert w1_activation.shape == (14336,), f"Unexpected shape for {key} in w=1: {w1_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w1'] = w1_activation


            w3_activation = w3[key].cpu().to(torch.float16).numpy()
            assert w3_activation.shape == (14336,), f"Unexpected shape for {key} in w=3: {w3_activation.shape}"
            repos_to_activations[f'{row["username/repo_name"]}.w3'] = w3_activation
        
        curr_expert_df = pl.DataFrame(repos_to_activations)
        curr_expert_df = curr_expert_df.with_columns(pl.lit(expert, dtype=pl.Int8).alias("expert"))
        curr_expert_df = curr_expert_df.select(pl.col(df.columns))


        df = pl.concat([df, curr_expert_df], how="vertical")
    
    layer_to_df[layer] = df

DuplicateError: could not create a new DataFrame: column with name 'but.w3' has more than one occurrence

In [ ]:
df = layer_to_df[28]


cols_end_with_w1_suffix = [col for col in df.columns if col.endswith('.w1')]
cols_end_with_w3_suffix = [col for col in df.columns if col.endswith('.w3')]

# df_w1_mean = df.select(*cols_end_with_w1_suffix).mean_horizontal()
df_w3_mean = df.select(*cols_end_with_w3_suffix).mean_horizontal()

# df_w1_mean.arg_sort(descending=True).head(10) // 14336

In [ ]:
df = layer_to_df[28]

df_top_50_indices = df.drop('expert').select(
    pl.all().arg_sort(descending=True).head(10)
)

df_top_50_values = df.drop('expert').select(
    pl.all().sort(descending=True).head(10)
)

df_top_50_values

# df_top_50_indices
df_top_50_indices // 14336

The.w3,Be.w3,To.w3,Of.w3,And.w3,A.w3,In.w3,That.w3,Have.w3,I.w3,It.w3,For.w3,Not.w3,On.w3,With.w3,He.w3,As.w3,You.w3,Do.w3,At.w3
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5
3,2,3,3,3,3,3,3,3,3,3,3,3,3,3,5,3,3,3,3
0,6,0,0,5,6,2,5,6,6,0,6,2,6,6,6,6,2,5,5
6,3,6,5,5,2,6,0,5,6,2,2,5,0,1,7,5,5,6,2
2,5,7,6,7,3,5,7,1,6,5,5,2,6,1,2,2,6,6,6
6,7,5,6,7,5,1,1,5,0,6,0,5,2,5,7,0,0,5,6
1,0,2,2,2,2,6,6,6,6,5,2,6,0,0,3,2,7,2,0
5,2,2,2,6,6,7,4,5,2,6,2,7,5,2,7,2,2,6,2
5,1,2,1,0,7,2,5,0,2,2,6,0,6,2,5,4,0,5,5


In [ ]:
fname = 'common_words.txt'

# read all lines not starting with #:
with open(fname, 'r') as f:
    lines = f.readlines()
common_words = [line.strip() for line in lines if not line.startswith('#')]
common_words

['the',
 'of',
 'and',
 'to',
 'a',
 'in',
 'that',
 'I',
 'was',
 'he',
 'his',
 'with',
 'is',
 'it',
 'for',
 'as',
 'had',
 'you',
 'not',
 'be',
 'on',
 'at',
 'by',
 'her',
 'which',
 'have',
 'or',
 'from',
 'this',
 'but',
 'all',
 'him',
 'she',
 'were',
 'they',
 'my',
 'are',
 'so',
 'me',
 'their',
 'an',
 'one',
 'de',
 'we',
 'who',
 'would',
 'said',
 'been',
 'no',
 'He',
 'will',
 'them',
 'when',
 'if',
 'there',
 'more',
 'out',
 'And',
 'It',
 'any',
 'up',
 'into',
 'your',
 'has',
 'do',
 'what',
 'could',
 'but',
 'our',
 'than',
 'other',
 'some',
 'very',
 'man',
 'upon',
 'about',
 'its',
 'only',
 'time',
 'may',
 'la',
 'like',
 'little',
 'then',
 'now',
 'should',
 'can',
 'made',
 'did',
 'such',
 'A',
 'great',
 'In',
 'must',
 'these',
 'two',
 'before',
 'see',
 'us',
 'over',
 'et',
 'much',
 'know',
 'after',
 'she',
 'first',
 'i',
 'good',
 'Mr',
 'down',
 'never',
 'most',
 'You',
 'where',
 'those',
 'old',
 'men',
 'own',
 'shall',
 'le',
 'came

In [ ]:
df_top_50_values

The.w3,Be.w3,To.w3,Of.w3,And.w3,A.w3,In.w3,That.w3,Have.w3,I.w3,It.w3,For.w3,Not.w3,On.w3,With.w3,He.w3,As.w3,You.w3,Do.w3,At.w3
f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16,f16
18.75,23.375,21.25,23.875,23.0,21.75,22.0,21.75,21.125,21.75,21.25,22.375,23.0,22.375,21.75,21.875,22.25,20.875,21.25,21.875
13.3125,11.0625,15.5625,16.125,14.6875,12.25,12.125,17.875,15.375,13.25,14.8125,16.125,13.25,14.6875,17.5,10.8125,17.0,12.875,13.8125,16.0
10.0,10.6875,12.125,15.625,11.5625,8.5,11.75,14.8125,13.4375,10.9375,12.0625,12.0,11.625,13.5,14.6875,10.5625,12.5,11.125,11.75,11.8125
9.625,10.25,11.0625,12.1875,11.125,8.0625,11.1875,14.0625,12.125,10.1875,11.625,11.875,10.9375,12.9375,12.9375,10.5,12.0,11.0625,10.875,11.1875
9.3125,10.1875,10.9375,11.375,10.5625,8.0,10.5625,11.5,11.8125,9.0625,11.125,11.5625,10.75,11.5,12.9375,10.1875,11.9375,10.75,10.8125,11.0625
9.125,9.4375,10.3125,11.1875,10.0,7.875,9.25,11.4375,10.625,8.4375,11.125,11.375,10.6875,11.375,12.0,10.0625,11.1875,10.5,10.75,10.5
8.625,9.25,10.0,10.9375,9.9375,7.8125,9.25,11.4375,10.5625,8.4375,10.75,11.375,10.5,10.9375,11.875,9.875,11.1875,10.3125,10.375,10.3125
8.375,9.125,9.875,10.875,9.6875,7.8125,9.25,11.0625,10.4375,8.3125,10.5625,10.9375,9.8125,10.375,10.5625,9.25,11.125,9.4375,9.6875,10.3125
8.375,9.0625,9.8125,10.8125,9.5625,7.8125,9.125,10.875,10.375,8.25,10.125,10.9375,9.3125,10.3125,10.5625,9.0,10.6875,9.1875,9.5,10.1875


In [ ]:
from pathlib import Path
import re

lines = Path('genesis.txt').read_text().splitlines()

chapter, verse, text = [], [], []
for line in lines:
    # [chapter:verse] text
    match = re.match(r'\[(\d+):(\d+)\] (.+)', line)
    if not match:
        raise ValueError(f"Line does not match expected format: {line}")
    chapter.append(int(match.group(1)))
    verse.append(int(match.group(2)))
    text.append(match.group(3))

df_genesis = pd.DataFrame({'chapter': chapter, 'verse': verse, 'text': text})
df_genesis.to_csv('genesis.csv', index=False)

FileNotFoundError: [Errno 2] No such file or directory: 'genesis.txt'

'So God made the dome and separated the waters that were under the dome from the waters that were above the dome. And it was so.'

In [ ]:
# join to chapter and text
df_genesis = pd.read_csv('data/genesis.csv')
df_genesis_by_chapter = df_genesis.groupby('chapter').apply(lambda g: g.sort_values('verse')['text'].tolist())
df_genesis_by_chapter = df_genesis_by_chapter.apply(lambda verses: '\n'.join(verses))
df_genesis_by_chapter.to_frame(name='text').reset_index()


/tmp/ipykernel_119397/1620948220.py:3: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_genesis_by_chapter = df_genesis.groupby('chapter').apply(lambda g: g.sort_values('verse')['text'].tolist())


,chapter,text
0,1,In the beginning when God created the heavens ...
1,2,"Thus the heavens and the earth were finished, ..."
2,3,Now the serpent was more crafty than any other...
3,4,"Now the man knew his wife Eve, and she conceiv..."
4,5,This is the list of the descendants of Adam. W...
5,6,When people began to multiply on the face of t...
6,7,"Then the LORD said to Noah, ""Go into the ark, ..."
7,8,But God remembered Noah and all the wild anima...
8,9,"God blessed Noah and his sons, and said to the..."
9,10,"These are the descendants of Noah's sons, Shem..."
